# PySlice TACAW, TEM, And STEM Tutorial

This notebook teaches the PySlice workflow used for phonon EELS and diffraction tasks:

1. load a real molecular-dynamics trajectory (or run one using the universal interatomic potentials),
2. propagate TEM or STEM probes with `MultisliceCalculator`,
3. inspect the returned `WFData`,
4. compute TACAW spectra, spectral diffraction, gain/loss branches, and temperature-corrected intensity,
5. compute HAADF-STEM images from focused-probe exit waves.

The core tutorial uses the hBN trajectory that the TACAW tests exercise. A small universal-potential MD example appears at the end as a generator pattern, but the TACAW analysis does not wait on a long MD run.

## Install And Launch

Run this notebook from an environment where PySlice is installed. The tutorial assumes `uv`:

```bash
uv sync --extra fast
uv sync --python 3.12 --extra fast --extra md
uv run --with jupyter --with ipywidgets jupyter lab example.ipynb
```

`fast` installs the PyTorch path used for GPU acceleration. The `md` extra installs ORB for the optional live MD generator section and should be synced with Python 3.12 because ORB's `dm-tree` dependency does not publish Python 3.13 wheels. The main loaded-trajectory TACAW and STEM sections do not require model downloads.

Parameter guidance is kept next to the API calls below. The comments are deliberately terse: change those values first when adapting the notebook to a different material, trajectory length, detector, or memory budget.

In [ ]:
from pathlib import Path
import os

try:
    import pyslice
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PySlice is not installed in this kernel. Run `uv sync --extra fast` "
        "or `uv sync --python 3.12 --extra fast --extra md`, "
        "then launch Jupyter with `uv run --with jupyter --with ipywidgets jupyter lab`."
    ) from exc

try:
    import torch
except ModuleNotFoundError:
    torch = None

import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk

from pyslice import (
    Loader,
    MultisliceCalculator,
    TACAWData,
    ORBMDCalculator,
    analyze_md_trajectory,
)

try:
    from ipywidgets import IntSlider, interact
    WIDGETS_AVAILABLE = True
except ModuleNotFoundError:
    WIDGETS_AVAILABLE = False

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "tests" and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "tests" / "inputs"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tutorial"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / "matplotlib"))

if torch is None:
    MULTISLICE_DEVICE = "cpu"
elif torch.cuda.is_available():
    MULTISLICE_DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    MULTISLICE_DEVICE = "mps"
else:
    MULTISLICE_DEVICE = "cpu"

MD_DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180})
print(f"PySlice version: {pyslice.__version__}")
print(f"PySlice multislice device: {MULTISLICE_DEVICE}")
print(f"MD generator device: {MD_DEVICE}")
print(f"Tutorial outputs: {OUTPUT_DIR}")
print(f"ipywidgets available: {WIDGETS_AVAILABLE}")


## Load A Real hBN Trajectory

`Loader` converts external atomistic data into a PySlice `Trajectory`.

For the LAMMPS trajectory below, `timestep=0.005` means adjacent saved frames are separated by 0.005 ps. That directly controls the TACAW frequency axis: total recorded time sets the frequency-bin spacing, and the saved-frame interval sets the Nyquist frequency.

For production TACAW, it is considered best practice to average over several trajectories to better sample the thermal ensemble and ensure ergodicity.

`atom_mapping={1: "B", 2: "N"}` converts LAMMPS type ids into element identities so the multislice potential can use the right scattering factors.

In [ ]:
A_LATTICE = 2.4907733333333337
B_LATTICE = 2.1570729817355123

trajectory = Loader(
    filename=str(INPUT_DIR / "hBN_truncated.lammpstrj"),  # LAMMPS dump to convert into a PySlice Trajectory.
    timestep=0.005,  # ps between saved frames; sets TACAW frequency resolution and Nyquist limit.
    atom_mapping={1: "B", 2: "N"},  # LAMMPS type id -> element symbol for scattering factors.
).load()

recorded_time_ps = trajectory.n_frames * trajectory.timestep
frequency_resolution_thz = 1.0 / recorded_time_ps
nyquist_thz = 0.5 / trajectory.timestep
lx, ly, lz = trajectory.extent

print(f"frames: {trajectory.n_frames}")
print(f"atoms/frame: {trajectory.n_atoms}")
print(f"saved-frame timestep: {trajectory.timestep:.4f} ps")
print(f"recorded time: {recorded_time_ps:.3f} ps")
print(f"frequency resolution: {frequency_resolution_thz:.2f} THz")
print(f"Nyquist frequency: {nyquist_thz:.1f} THz")
print(f"box extent: {lx:.1f} x {ly:.1f} x {lz:.1f} Angstrom")

## Load A Static Structure From CIF

CIF files are single-structure inputs, so `Loader` reads them through ASE and returns a one-frame `Trajectory`. Tile the unit cell before multislice if the crystallographic cell is too small for the probe or detector geometry.


In [ ]:
cif_trajectory = Loader(filename=str(INPUT_DIR / "hBN_cif.cif")).load()
cif_supercell = cif_trajectory.tile_positions((4, 4, 1))
cif_frozen = cif_supercell.generate_random_displacements(
    n_displacements=4,  # independent static snapshots for a compact frozen-phonon demo.
    sigma=0.05,  # Angstrom RMS displacement around the CIF positions.
    seed=3,
)

print(f"CIF frames: {cif_trajectory.n_frames}")
print(f"CIF atoms/cell: {cif_trajectory.n_atoms}")
print(f"tiled atoms/frame: {cif_supercell.n_atoms}")
print(f"frozen-phonon frames from CIF: {cif_frozen.n_frames}")


## Optional: Generate A Small Universal-Potential MD Trajectory

The loaded hBN dump is the main trajectory used for TACAW below. This optional section shows the other common path: use a universal machine-learning potential to generate an ASE trajectory, then pass the result into the same `MultisliceCalculator` and `TACAWData` APIs.

The example uses ORB on a tiny silicon cell: first a zero-temperature position-and-cell relaxation, then finite-temperature equilibration, then a short NVE production run. CUDA is used when available. On Apple Silicon, ORB's periodic neighbor-list dependency currently does not support MPS, so the MD generator uses CPU while PySlice multislice can still use MPS.

In [ ]:
md_output_dir = OUTPUT_DIR / "md_orb_si_tiny_cell_relaxed"
md_output_dir.mkdir(parents=True, exist_ok=True)
trajectory_file = md_output_dir / "production.traj"
log_file = md_output_dir / "production.log"

if trajectory_file.exists() and (md_output_dir / "relaxed_structure.xyz").exists():
    generated_trajectory = Loader(
        filename=str(trajectory_file),  # ASE trajectory written by the ORB MD run.
        timestep=0.004,  # ps between saved frames: 2 fs timestep x save_interval=2.
    ).load()
    print(f"loaded cached ORB trajectory: {trajectory_file}")
elif trajectory_file.exists():
    print("cached ORB trajectory has no relaxation record; regenerating it")
    trajectory_file.unlink()

if not trajectory_file.exists():
    atoms = bulk("Si", crystalstructure="diamond", a=5.431, cubic=True)
    print(f"initial ORB structure: {len(atoms)} atoms")

    md = ORBMDCalculator(
        model_name="orb-v3-conservative-inf-omat",  # conservative ORB model appropriate for force-based MD.
        device=MD_DEVICE,  # use cuda when available; otherwise cpu for ORB compatibility.
        weights_path=os.environ.get("ORB_WEIGHTS_PATH"),  # optional local model weights path.
    )
    atoms = md.relax_structure(
        atoms,
        fmax=0.05,  # eV/Angstrom force tolerance; lower is stricter and slower.
        steps=80,  # optimizer step cap for this small demonstration.
        optimizer="FIRE",  # ASE optimizer choice: "FIRE", "BFGS", or "LBFGS".
        output_dir=md_output_dir,  # writes relaxed_structure.xyz and optimizer logs.
        relax_cell=True,  # relax atomic positions and cell vectors; use False for fixed-cell relaxation.
    )
    print(f"relaxed cell lengths: {atoms.cell.lengths().round(3)} Angstrom")
    print(f"relaxed potential energy: {atoms.get_potential_energy() / len(atoms):.4f} eV/atom")

    md.setup(
        atoms=atoms,  # relaxed ASE Atoms object used as the MD initial condition.
        temperature=300,  # K target for velocity initialization and NVT equilibration.
        timestep=2.0,  # fs MD integration step.
        ensemble="nvt",  # thermostat ensemble used during equilibration.
        friction=0.05,  # Langevin friction for NVT equilibration; larger damps faster.
        min_equilibration_steps=10,  # minimum equilibration steps before convergence checks can pass.
        max_equilibration_steps=40,  # hard cap so this tutorial cannot run indefinitely.
        check_interval=10,  # steps between temperature/energy convergence checks.
        production_ensemble="nve",  # microcanonical production after equilibration.
        production_steps=40,  # short tutorial run; increase substantially for real TACAW statistics.
        save_interval=2,  # save every 2 MD steps, so saved-frame timestep is 0.004 ps.
        output_dir=md_output_dir,  # production.traj and production.log destination.
        save_xyz=True,  # also write xyz snapshots for quick external inspection.
        rng=np.random.default_rng(7),  # deterministic initial velocities for a reproducible tutorial run.
    )
    generated_trajectory = md.run()

print(f"generated frames: {generated_trajectory.n_frames}")
print(f"generated atoms/frame: {generated_trajectory.n_atoms}")
print(f"generated timestep: {generated_trajectory.timestep:.4f} ps")

## MD Diagnostics

`analyze_md_trajectory()` makes four plots for MD sanity checks:

- **Temperature evolution**: whether instantaneous temperature fluctuates around the target instead of drifting away.
- **Energy evolution**: potential, kinetic, and total energy; in NVE production, total energy drift is the warning sign.
- **RMSD from initial structure**: a coarse structural-stability check relative to the first saved frame.
- **Potential-energy distribution**: whether sampled configurations look like one stationary state or a mixed/poorly equilibrated set.

These diagnostics do not prove the trajectory is physically sufficient for TACAW, but they catch many bad trajectories before electron scattering.

In [ ]:
orb_dir = OUTPUT_DIR / "md_orb_si_tiny_cell_relaxed"

if (orb_dir / "production.traj").exists() and (orb_dir / "production.log").exists():
    analyze_md_trajectory(
        trajectory_file=str(orb_dir / "production.traj"),  # ASE trajectory to inspect.
        log_file=str(orb_dir / "production.log"),  # MD log containing temperature and energy traces.
        skip_frames=1,  # analyze every saved frame; increase for very long trajectories.
        output_file=str(orb_dir / "md_analysis.png"),  # diagnostic figure written next to the trajectory.
    )
else:
    print("Run the optional ORB MD cell first to generate diagnostics.")


## Prepare A Full Orthogonal Cell

The hBN dump is a triclinic LAMMPS cell, so some Cartesian x positions are negative even though they are periodic images of atoms in the same sheet. For TACAW, keep the full in-plane cell: the reciprocal-space pixel spacing is set by the real-space box length. Spatially cropping atoms improves speed, but it coarsens or distorts the physical k-grid unless the new cell is constructed with care.

`fold_positions_to_orthogonal_box(axes=(0,))` folds the periodic x coordinates back into the positive side of the cell and replaces the tilted box metadata with an orthogonal box using the original diagonal lengths. No atoms or time frames are removed.

In [ ]:
analysis_trajectory = trajectory.fold_positions_to_orthogonal_box(
    axes=(0,),  # fold periodic x coordinates into a positive orthogonal box; keep y and z unchanged.
)
analysis_lx, analysis_ly, analysis_lz = np.diag(analysis_trajectory.box_matrix)
dkx = 1.0 / analysis_lx
dky = 1.0 / analysis_ly

print(f"atoms/frame: {analysis_trajectory.n_atoms}")
print(f"orthogonal box: {analysis_lx:.2f} x {analysis_ly:.2f} x {analysis_lz:.2f} Angstrom")
print(f"frames kept for TACAW: {analysis_trajectory.n_frames}")
print(f"k spacing: dkx={dkx:.4f}, dky={dky:.4f} 1/Angstrom")
print(f"x range after folding: {analysis_trajectory.positions[:, :, 0].min():.3f} to {analysis_trajectory.positions[:, :, 0].max():.3f} Angstrom")

analysis_trajectory.plot(timestep=0, view="xy", size=5)

## TEM Diffraction With `MultisliceCalculator`

A parallel beam uses `aperture=0`. That is the natural TEM/TACAW setup: one incident wave is propagated through every saved MD frame, producing a `WFData` array with layout `(probe, time, kx, ky, layer)`.

The comments in `setup()` are the practical tuning guide: real-space box length sets reciprocal-space spacing, `sampling` sets the simulation pixel size and reachable k range, and `max_kx`/`max_ky` crop what is stored after propagation.

In [ ]:
TEM_SAMPLING_A = 0.1
TEM_SLICE_THICKNESS_A = 0.5
TEM_MAX_K = 2.0

tem_calc = MultisliceCalculator(device=MULTISLICE_DEVICE)
tem_calc.setup(
    analysis_trajectory,  # full folded hBN trajectory; all frames are propagated for TACAW.
    aperture=0,  # 0 mrad = parallel-beam TEM. Use nonzero mrad for focused STEM probes.
    voltage_eV=100e3,  # accelerating voltage; controls electron wavelength and aperture-to-k conversion.
    sampling=TEM_SAMPLING_A,  # real-space pixel size in Angstrom; smaller is more accurate but more expensive.
    slice_thickness=TEM_SLICE_THICKNESS_A,  # Angstrom thickness of projected-potential slices along the beam.
    max_kx=TEM_MAX_K,  # stored reciprocal crop in 1/Angstrom; does not change k spacing.
    max_ky=TEM_MAX_K,  # use symmetric crops unless the detector geometry is anisotropic.
    return_layers=-1,
    cache_wavefunctions=True,  # keep exit waves because TACAW needs the full time series.
    use_memmap=False,  # set True for large WFData arrays that should live on disk instead of RAM.
)
wf_tem = tem_calc.run()

print(f"WFData shape: {tuple(wf_tem.array.shape)}")
print("layout: probe, time, kx, ky, layer")
print(f"k-grid: {len(wf_tem.kxs)} x {len(wf_tem.kys)}")
print(f"cache directory: {wf_tem.cache_dir}")

In [ ]:
wf_tem.plot_reciprocal(
    filename=OUTPUT_DIR / "02_tem_diffraction.png",
    whichProbe="mean",
    whichTimestep="mean",
    powerscaling=0.25,
    extent=(-TEM_MAX_K, TEM_MAX_K, -TEM_MAX_K, TEM_MAX_K),
    nuke_zerobeam=True,
    title="Parallel-beam diffraction from hBN trajectory",
)
plt.show()

## Frozen-Phonon Multislice From A Static Structure

TACAW uses a time-ordered MD trajectory. Conventional frozen phonon is different: start from one static structure, draw independent random displacements, and average ordinary multislice exit waves over that ensemble.

This example builds a modest silicon crystal from scratch with ASE, converts it to a PySlice `Trajectory`, then calls `generate_random_displacements()`. The supercell is intentionally smaller than the hBN TACAW cell: frozen phonon is here to show the static-structure API, not to dominate the tutorial runtime.

In [ ]:
static_trajectory = Loader(
    atoms=bulk("Si", crystalstructure="diamond", a=5.431, cubic=True).repeat((8, 8, 2)),  # ASE Atoms -> one-frame static Trajectory.
).load()
frozen_trajectory = static_trajectory.generate_random_displacements(
    n_displacements=8,  # number of random configurations to average; increase for smoother frozen-phonon results.
    sigma=0.08,  # Gaussian displacement width in Angstrom.
    seed=11,  # reproducible random displacements.
)

frozen_calc = MultisliceCalculator(device=MULTISLICE_DEVICE)
frozen_calc.setup(
    frozen_trajectory,  # independent displaced static structures.
    aperture=0,  # parallel-beam TEM frozen-phonon calculation.
    voltage_eV=100e3,  # match the TEM/TACAW voltage for comparison.
    sampling=TEM_SAMPLING_A,  # reuse the TEM pixel size.
    slice_thickness=TEM_SLICE_THICKNESS_A,  # reuse the TEM slice thickness.
    max_kx=TEM_MAX_K,  # reciprocal crop for the saved diffraction pattern.
    max_ky=TEM_MAX_K,
    cache_wavefunctions=False,  # no cache needed for this small one-off demonstration.
)
wf_frozen = frozen_calc.run()

print(f"static structure atoms: {static_trajectory.n_atoms}")
print(f"frozen-phonon configurations: {frozen_trajectory.n_frames}")
print(f"frozen WFData shape: {tuple(wf_frozen.array.shape)}")

wf_frozen.plot_reciprocal(
    filename=OUTPUT_DIR / "03_frozen_phonon_diffraction.png",
    whichProbe="mean",
    whichTimestep="mean",
    powerscaling=0.25,
    extent=(-TEM_MAX_K, TEM_MAX_K, -TEM_MAX_K, TEM_MAX_K),
    nuke_zerobeam=True,
    title="Frozen-phonon diffraction, sigma=0.08 Angstrom",
)
plt.show()

## TACAW From TEM Exit Waves

`TACAWData` Fourier-transforms the time-domain exit waves into frequency-resolved intensity. Positive frequencies are the loss branch; negative frequencies are the gain branch.

For real data, set `temperature_K` and leave `apply_bose=True` so the TACAW intensity carries the Bose-factor detailed-balance correction. Use `chunkFFT=True` when the time FFT is too large to do as one dense operation.

In [ ]:
SAMPLE_TEMPERATURE_K = 300.0
LOSS_FREQUENCY_THZ = 30.0
GAIN_FREQUENCY_THZ = -LOSS_FREQUENCY_THZ

tacaw = TACAWData(
    wf_tem,  # TEM exit-wave time series from MultisliceCalculator.
    temperature_K=SAMPLE_TEMPERATURE_K,  # sample temperature for Bose-factor correction.
    apply_bose=True,  # TACAW should be Bose corrected for gain/loss detailed balance.
    chunkFFT=False,  # set True for very large WFData arrays when the full time FFT is too large.
)

frequencies = np.asarray(tacaw.frequencies)
spectrum = tacaw.spectrum(probe_index=None)  # None averages over all probe positions; TEM has one probe.

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(frequencies, spectrum, color="black", linewidth=1.4)
ax.axvline(0.0, color="0.55", linewidth=0.8)
ax.axvline(LOSS_FREQUENCY_THZ, color="#4c72b0", linestyle="--", linewidth=1.0, label=f"loss {LOSS_FREQUENCY_THZ:.1f} THz")
ax.set_xlabel("frequency (THz); negative = gain, positive = loss")
ax.set_ylabel("integrated intensity")
ax.set_title(f"TACAW spectrum, {SAMPLE_TEMPERATURE_K:.0f} K")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_tacaw_spectrum.png")
plt.show()

## Frequency-Integrated And Spectral TACAW Diffraction

`diffraction()` sums TACAW intensity over all frequencies. `spectral_diffraction(frequency)` selects the nearest signed frequency bin and returns the corresponding diffraction map.

In [ ]:
integrated_diffraction = tacaw.diffraction(probe_index=None)  # frequency-integrated TACAW diffraction.
loss_diffraction = tacaw.spectral_diffraction(LOSS_FREQUENCY_THZ, probe_index=None)  # nearest positive-frequency loss bin.
gain_diffraction = tacaw.spectral_diffraction(GAIN_FREQUENCY_THZ, probe_index=None)  # nearest negative-frequency gain bin.

tacaw.plot(
    integrated_diffraction ** 0.20,  # power scaling makes weak diffuse intensity visible.
    "kx",
    "ky",
    filename=OUTPUT_DIR / "05_tacaw_integrated_diffraction.png",
    title="TACAW diffraction summed over frequency",
)

tacaw.plot(
    loss_diffraction ** 0.20,
    "kx",
    "ky",
    filename=OUTPUT_DIR / "05_tacaw_loss_diffraction.png",
    title=f"Spectral Diffraction: {LOSS_FREQUENCY_THZ:.1f} THz",
)


## Interactive Spectral Diffraction

The slider passes a signed frequency-bin index to `spectral_diffraction()`. Use this to scan the TACAW diffraction pattern through gain, zero, and loss frequencies without changing the underlying calculation.

In [ ]:
def show_spectral_diffraction(frequency_index):
    frequency_index = int(frequency_index)
    frequency = float(frequencies[frequency_index])
    tacaw.plot(
        tacaw.spectral_diffraction(frequency, probe_index=None) ** 0.20,
        "kx",
        "ky",
        title=f"TACAW spectral diffraction: {frequency:.1f} THz",
    )

default_frequency_index = int(np.argmin(np.abs(frequencies - LOSS_FREQUENCY_THZ)))

if WIDGETS_AVAILABLE:
    frequency_slider = IntSlider(
        value=default_frequency_index,
        min=0,
        max=len(frequencies) - 1,
        step=1,
        description="frequency bin",
        continuous_update=False,
    )
    interact(show_spectral_diffraction, frequency_index=frequency_slider);
else:
    show_spectral_diffraction(default_frequency_index)

## Focused-Probe 2G And -2G Masks

`masked_spectrum()` integrates a selected reciprocal-space region. For a physically meaningful detector condition, run a small focused probe and center the masks on diffraction conditions instead of using a generic low-q disk.

This cell runs a single 5 mrad probe at the center of the hBN cell, builds TACAW from that focused-probe `WFData`, then selects circular masks around `+2G` and `-2G`. The mask radius is the 5 mrad convergence semi-angle converted to reciprocal-space units using the probe wavelength.

In [ ]:
probe_5mrad_calc = MultisliceCalculator(device=MULTISLICE_DEVICE)
probe_5mrad_calc.setup(
    analysis_trajectory,  # same full folded hBN trajectory used for the parallel-beam TACAW run.
    aperture=5,  # mrad convergence semi-angle; small enough to isolate +/-2G disks cleanly.
    voltage_eV=100e3,  # match the parallel-beam TACAW voltage.
    sampling=TEM_SAMPLING_A,  # reuse the TEM sampling so the reciprocal grid is comparable.
    slice_thickness=TEM_SLICE_THICKNESS_A,
    probe_xs=[analysis_lx / 2],  # one probe at the center of the orthogonal cell.
    probe_ys=[analysis_ly / 2],
    max_kx=TEM_MAX_K,
    max_ky=TEM_MAX_K,
    cache_wavefunctions=False,  # no cache needed for this one-probe condition-selection run.
)
wf_5mrad = probe_5mrad_calc.run()

tacaw_5mrad = TACAWData(
    wf_5mrad,
    temperature_K=SAMPLE_TEMPERATURE_K,
    apply_bose=True,
    chunkFFT=False,
)
probe_frequencies = np.asarray(tacaw_5mrad.frequencies)

two_g_kx = 2.0 / A_LATTICE  # PySlice k axes are in 1/Angstrom, so Gx = 1 / a for this cell convention.
condition_radius = 5e-3 / tacaw_5mrad.probe.wavelength  # 5 mrad convergence disk radius in 1/Angstrom.

plus_2g_spectrum = tacaw_5mrad.masked_spectrum(
    {"shape": "round", "center": (two_g_kx, 0.0), "radius": condition_radius},
    probe_index=0,
)
minus_2g_spectrum = tacaw_5mrad.masked_spectrum(
    {"shape": "round", "center": (-two_g_kx, 0.0), "radius": condition_radius},
    probe_index=0,
)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(probe_frequencies, plus_2g_spectrum, color="green", linewidth=1.2, label="+2G condition")
ax.plot(probe_frequencies, minus_2g_spectrum, color="blue", linewidth=1.2, label="-2G condition")
ax.axvline(0.0, color="0.55", linewidth=0.8)
ax.axvline(LOSS_FREQUENCY_THZ, color="0.35", linestyle="--", linewidth=0.9)
ax.set_xlabel("frequency (THz)")
ax.set_ylabel("integrated intensity")
ax.set_title("5 mrad TACAW spectra from +/-2G masks")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_tacaw_5mrad_2g_spectra.png")
plt.show()

condition_map = tacaw_5mrad.diffraction(probe_index=0)  # summed over all TACAW frequency bins.
kxs_5mrad = np.asarray(tacaw_5mrad.kxs)
kys_5mrad = np.asarray(tacaw_5mrad.kys)
theta = np.linspace(0.0, 2.0 * np.pi, 240)
mask_extent = two_g_kx + 2.0 * condition_radius

fig, ax = plt.subplots(figsize=(5.6, 5.6))
mesh = ax.pcolormesh(kxs_5mrad, kys_5mrad, condition_map.T ** 0.20, shading="auto", cmap="inferno")
for center, color, label in [
    ((two_g_kx, 0.0), "green", "+2G"),
    ((-two_g_kx, 0.0), "blue", "-2G"),
]:
    ax.plot(
        center[0] + condition_radius * np.cos(theta),
        center[1] + condition_radius * np.sin(theta),
        color=color,
        linewidth=1.3,
        label=label,
    )
ax.set_xlim(-mask_extent, mask_extent)
ax.set_ylim(-mask_extent, mask_extent)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("kx (1/Angstrom)")
ax.set_ylabel("ky (1/Angstrom)")
ax.set_title("5 mrad TACAW diffraction summed over frequency")
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.colorbar(mesh, ax=ax, label="intensity^0.2")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_tacaw_5mrad_2g_masks.png")
plt.show()

print(f"5 mrad WFData shape: {tuple(wf_5mrad.array.shape)}")
print(f"+/-2G centers: +/-{two_g_kx:.3f} 1/Angstrom")
print(f"condition-mask radius: {condition_radius:.3f} 1/Angstrom")

## Interactive 2G Collection With Frequency

The aperture disks are fixed detector conditions. This slider changes only the TACAW frequency slice shown underneath them, so the collected signal can be compared with the masked spectra above.

In [ ]:
def show_2g_collection(frequency_index):
    frequency_index = int(frequency_index)
    frequency = float(probe_frequencies[frequency_index])
    frequency_map = tacaw_5mrad.spectral_diffraction(frequency, probe_index=0)

    fig, ax = plt.subplots(figsize=(5.6, 5.6))
    mesh = ax.pcolormesh(kxs_5mrad, kys_5mrad, frequency_map.T ** 0.20, shading="auto", cmap="inferno")
    for center, color, label in [
        ((two_g_kx, 0.0), "#4c72b0", "+2G"),
        ((-two_g_kx, 0.0), "#dd8452", "-2G"),
    ]:
        ax.plot(
            center[0] + condition_radius * np.cos(theta),
            center[1] + condition_radius * np.sin(theta),
            color=color,
            linewidth=1.3,
            label=label,
        )
    ax.set_xlim(-mask_extent, mask_extent)
    ax.set_ylim(-mask_extent, mask_extent)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("kx (1/Angstrom)")
    ax.set_ylabel("ky (1/Angstrom)")
    ax.set_title(f"5 mrad TACAW collection: {frequency:.1f} THz")
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    fig.colorbar(mesh, ax=ax, label="intensity^0.2")
    fig.tight_layout()
    plt.show()

default_collection_index = int(np.argmin(np.abs(probe_frequencies)))

if WIDGETS_AVAILABLE:
    collection_slider = IntSlider(
        value=default_collection_index,
        min=0,
        max=len(probe_frequencies) - 1,
        step=1,
        description="frequency bin",
        continuous_update=False,
    )
    interact(show_2g_collection, frequency_index=collection_slider);
else:
    show_2g_collection(default_collection_index)

## Momentum-Resolved Dispersion Cut

`dispersion()` samples intensity along a requested path through momentum space and returns an array with shape `(frequency, path_position)`. This uses the parallel-beam TACAW object from the main TEM run, which is the cleaner geometry for a one-dimensional momentum cut.

In [ ]:
kx_path = np.linspace(0.0, min(1 / A_LATTICE, float(np.max(tacaw.kxs))), 120)  # momentum path from Gamma along +kx.
ky_path = np.zeros_like(kx_path)  # ky=0 keeps the cut on a high-symmetry line for this example.
dispersion = tacaw.dispersion(kx_path, ky_path, probe_index=None)  # returns frequency bins x path points.

fig, ax = plt.subplots(figsize=(6.4, 4.0))
mesh = ax.pcolormesh(kx_path, frequencies, dispersion ** 0.50, shading="auto", cmap="inferno")
ax.axhline(0.0, color="white", linewidth=0.7, alpha=0.8)
ax.set_xlim(float(kx_path[0]), float(kx_path[-1]))
ax.set_ylim(float(frequencies[0]), float(frequencies[-1]))
ax.set_xlabel("kx (1/Angstrom)")
ax.set_ylabel("frequency (THz)")
ax.set_title("TACAW dispersion cut")
fig.colorbar(mesh, ax=ax, label="intensity^0.5")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_tacaw_dispersion.png")
plt.show()

print(f"dispersion shape: {dispersion.shape} = frequency bins x k-path points")
print(f"frequency range: {frequencies[0]:.1f} to {frequencies[-1]:.1f} THz")

## STEM And HAADF

For STEM, pass a real-space scan grid with `probe_xs` and `probe_ys`, and use a nonzero `aperture` to create focused probes. This section demonstrates the memory-efficient HAADF path: `ADF=(inner_mrad, outer_mrad), return_layers=None` integrates the annular detector during propagation instead of storing the full focused-probe diffraction datacube.

The next section runs a much smaller stored-stack STEM calculation for interactive diffraction-pattern inspection.

In [ ]:
STEM_SAMPLING_A = 0.25

stem_trajectory = analysis_trajectory.slice_timesteps(0, 1)  # one static frame is enough for the HAADF API demo.

probe_xs = np.linspace(1.0 * A_LATTICE, 3.0 * A_LATTICE, 24)  # STEM scan x positions in Angstrom.
probe_ys = np.linspace(1.0 * B_LATTICE, 3.0 * B_LATTICE, 24)  # STEM scan y positions in Angstrom.

stem_calc = MultisliceCalculator(device=MULTISLICE_DEVICE)
stem_calc.setup(
    stem_trajectory,  # one-frame trajectory for a static HAADF image.
    aperture=30,  # mrad convergence semi-angle for focused STEM probes.
    voltage_eV=100e3,  # match the TEM voltage.
    sampling=STEM_SAMPLING_A,  # coarser than TEM to keep a dense probe scan cheap.
    slice_thickness=TEM_SLICE_THICKNESS_A,  # use the same projected-potential slice thickness.
    probe_xs=probe_xs,  # x scan coordinates; len(probe_xs) x len(probe_ys) probes are evaluated.
    probe_ys=probe_ys,  # y scan coordinates.
    max_kx=2.0,  # reciprocal range retained for the on-the-fly ADF detector.
    max_ky=2.0,
    loop_probes=32,  # process focused probes in GPU-memory-friendly chunks. Increase/Decrease for compute/speed tradeoff
    cache_wavefunctions=False,  # no exit-wave cache because this cell only needs HAADF intensity.
    ADF=(45, 150),  # inner and outer annular detector angles in mrad.
    return_layers=None,  # skip the full 4D-STEM cube and return the compact HAADF image.
)
wf_stem, haadf = stem_calc.run()

print(f"STEM frames: {stem_trajectory.n_frames}")
print(f"probe grid: {len(probe_xs)} x {len(probe_ys)} = {len(wf_stem.probe_positions)} positions")
print("ADF detector: 45-150 mrad")
print(f"stored WFData shape in ADF mode: {tuple(wf_stem.array.shape)}")
print(f"HAADF image shape: {haadf.array.shape}")

In [ ]:
haadf.plot(filename=OUTPUT_DIR / "07_haadf_stem.png", title="HAADF-STEM image")
plt.show()

## STEM Probe Aberrations

After `setup()`, the focused probe is available as `calculator.base_probe`. Apply microscope aberrations with `base_probe.aberrate({...})` before `run()`. Coefficients use the `Cnm` notation used by the probe aberration helper; scalar values have zero azimuth, while `(value, angle)` pairs set an oriented aberration.


In [ ]:
aberration_probe_xs = np.linspace(1.0 * A_LATTICE, 3.0 * A_LATTICE, 24)
aberration_probe_ys = np.linspace(1.0 * B_LATTICE, 3.0 * B_LATTICE, 24)

aberrated_calc = MultisliceCalculator(device=MULTISLICE_DEVICE)
aberrated_calc.setup(
    stem_trajectory,
    aperture=30,
    voltage_eV=100e3,
    sampling=STEM_SAMPLING_A,
    slice_thickness=TEM_SLICE_THICKNESS_A,
    probe_xs=aberration_probe_xs,
    probe_ys=aberration_probe_ys,
    max_kx=2.0,
    max_ky=2.0,
    loop_probes=32,
    cache_wavefunctions=False,
    ADF=(45, 150),
    return_layers=None,
)
aberrated_calc.base_probe.aberrate({
    "C10": -100.0,  # defocus-like phase term, in Angstrom.
    "C12": (50.0, 0.0),  # two-fold astigmatism with orientation angle in radians.
    "C30": 1.0e4,  # spherical aberration.
})
wf_aberrated, haadf_aberrated = aberrated_calc.run()

haadf_aberrated.plot(
    filename=OUTPUT_DIR / "08_haadf_stem_aberrated.png",
    title="Aberrated HAADF-STEM image",
)
plt.show()

print(f"aberrated probe grid: {len(aberration_probe_xs)} x {len(aberration_probe_ys)}")
print(f"returned wavefunction layers: {wf_aberrated.layer.tolist()}")
print(f"aberrated HAADF image shape: {haadf_aberrated.array.shape}")


## Interactive STEM Diffraction

The HAADF cell intentionally avoids storing every focused-probe diffraction pattern. When you do want to inspect those patterns directly, run a smaller stored-stack calculation and move through probe positions with a slider.

In [ ]:
STEM_DIFFRACTION_MAX_K = 1.2
diff_probe_xs = np.linspace(1.0 * A_LATTICE, 3.0 * A_LATTICE, 5)  # smaller scan for stored diffraction patterns.
diff_probe_ys = np.linspace(1.0 * B_LATTICE, 3.0 * B_LATTICE, 5)

diff_calc = MultisliceCalculator(device=MULTISLICE_DEVICE)
diff_calc.setup(
    stem_trajectory,  # same one-frame STEM structure.
    aperture=30,  # focused STEM probe.
    voltage_eV=100e3,
    sampling=STEM_SAMPLING_A,
    slice_thickness=TEM_SLICE_THICKNESS_A,
    probe_xs=diff_probe_xs,  # 5 x positions gives 25 stored diffraction patterns.
    probe_ys=diff_probe_ys,
    max_kx=STEM_DIFFRACTION_MAX_K,  # smaller reciprocal crop keeps the stored datacube light.
    max_ky=STEM_DIFFRACTION_MAX_K,
    loop_probes=2,  # small chunks keep memory low while storing the full stack.
    cache_wavefunctions=False,
)
wf_stem_diffraction = diff_calc.run()


def plot_stem_probe_diffraction(probe_index):
    probe_index = int(probe_index)
    x, y = wf_stem_diffraction.probe_positions[probe_index]
    wf_stem_diffraction.plot_reciprocal(
        whichProbe=probe_index,
        whichTimestep="mean",
        powerscaling=0.12,
        extent=(-STEM_DIFFRACTION_MAX_K, STEM_DIFFRACTION_MAX_K, -STEM_DIFFRACTION_MAX_K, STEM_DIFFRACTION_MAX_K),
        nuke_zerobeam=True,
        title=f"Probe {probe_index}: x={x:.2f}, y={y:.2f} Angstrom",
    )

if WIDGETS_AVAILABLE:
    probe_slider = IntSlider(
        value=len(wf_stem_diffraction.probe_positions) // 2,
        min=0,
        max=len(wf_stem_diffraction.probe_positions) - 1,
        step=1,
        description="probe",
        continuous_update=False,
    )
    interact(plot_stem_probe_diffraction, probe_index=probe_slider);
else:
    plot_stem_probe_diffraction(len(wf_stem_diffraction.probe_positions) // 2)

## Summary

Loaded-trajectory TACAW path:

```python
trajectory = Loader("trajectory.lammpstrj", timestep=0.005, atom_mapping={1: "B", 2: "N"}).load()
trajectory = trajectory.fold_positions_to_orthogonal_box(axes=(0,))
calc = MultisliceCalculator(device="cuda")
calc.setup(trajectory, aperture=0, voltage_eV=100e3, sampling=0.2, slice_thickness=0.5, use_memmap=False)
wf = calc.run()
tacaw = TACAWData(wf, temperature_K=300, apply_bose=True, chunkFFT=False)
```

ORB-generated trajectory path:

```python
atoms = bulk("Si", crystalstructure="diamond", a=5.431, cubic=True)
md = ORBMDCalculator(model_name="orb-v3-conservative-inf-omat", device="cuda")
atoms = md.relax_structure(atoms, fmax=0.05, steps=80, optimizer="FIRE", relax_cell=True, output_dir="md_orb_si")
md.setup(atoms=atoms, temperature=300, timestep=2.0, production_ensemble="nve")
trajectory = md.run()
```

Frozen-phonon static multislice path:

```python
atoms = bulk("Si", crystalstructure="diamond", a=5.431, cubic=True).repeat((8, 8, 2))
static = Loader(atoms=atoms).load()
frozen = static.generate_random_displacements(n_displacements=8, sigma=0.08, seed=1)
calc = MultisliceCalculator(device="cuda")
calc.setup(frozen, aperture=0, voltage_eV=100e3, sampling=0.2, slice_thickness=0.5)
wf = calc.run()
```

For HAADF-STEM, use a focused aperture with `probe_xs`/`probe_ys` and prefer `ADF=(inner_mrad, outer_mrad), return_layers=None` when you only need the annular detector image. Store the full STEM diffraction cube only when you need 4D-STEM patterns or probe-by-probe reciprocal-space inspection.